# Cluster Analysis: Poetic Features

In [17]:
import warnings

warnings.filterwarnings("ignore")

from mqdq import babble, elegy
from mqdq import line_analyzer as la


import numpy as np
import pandas as pd


from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from typing import Callable, Collection, Union

import glob
import os

from pathlib import Path

In [19]:

def find_metrical_root():
    current = Path.cwd().resolve()

    for folder in [current, *current.parents]:
        if (folder / "corpus").is_dir():
            return folder

    raise FileNotFoundError("Metrical analysis root not found.")


BASE_DIR = find_metrical_root()
CORPUS_DIR = BASE_DIR / "corpus"

excluded_files = {
    "X3_machabeis.xml",
    "X4_certamina.xml",
}

files = sorted([
    fn for fn in CORPUS_DIR.glob("*.xml")
    if fn.name not in excluded_files
])

collection = []

for fn in files:
    parts = fn.stem.split("_")

    author = parts[0]
    title = parts[1]

    print("Sto leggendo:", fn.name)

    b = babble.Babbler.from_file(fn, name=title)
    b.author = author

    collection.append(b)

Sto leggendo: AI_anticlaudianus.xml
Lines: 4438 Corrupt: 43 Total Dropped: 43
Sto leggendo: AeC_medicaminum.xml
Lines: 4588 Corrupt: 174 Total Dropped: 174
Sto leggendo: AeC_viaticus.xml
Lines: 2373 Corrupt: 61 Total Dropped: 61
Sto leggendo: AeP_karolinus.xml
Lines: 2598 Corrupt: 89 Total Dropped: 89
Sto leggendo: BC_contemptu.xml
Lines: 2966 Corrupt: 62 Total Dropped: 62
Sto leggendo: B_judas.xml
Lines: 912 Corrupt: 55 Total Dropped: 55
Sto leggendo: DC_mathildis.xml
Lines: 2938 Corrupt: 142 Total Dropped: 142
Sto leggendo: E_biblia.xml
Lines: 1464 Corrupt: 56 Total Dropped: 56
Sto leggendo: FB_nuptis.xml
Lines: 4738 Corrupt: 103 Total Dropped: 103
Sto leggendo: GAp_roberti.xml
Lines: 2833 Corrupt: 59 Total Dropped: 59
Sto leggendo: GAr_philippis.xml
Lines: 1403 Corrupt: 52 Total Dropped: 52
Sto leggendo: GC_alexandreis.xml
Lines: 5511 Corrupt: 140 Total Dropped: 140
Sto leggendo: GM_merlini.xml
Lines: 1529 Corrupt: 37 Total Dropped: 37
Sto leggendo: GP_ligurinus.xml
Lines: 6575 Corr

In [ ]:
for b in collection:
    print(f"Author: {b.author}, Name: {b.name}")

In [ ]:
sz = 150 
collection_chunked = []  

for b in collection:
    ll = b.raw_source
    chunks = [ll[i : i + sz] for i in range(0, len(ll), sz)]

    for i, chunk in enumerate(chunks):
        if len(chunk) == sz:  
            new_babbler = babble.Babbler(
                chunk, name=f"{b.name} - {i+1}"
            )  
            new_babbler.author = b.author  
            collection_chunked.append(new_babbler)

print([f"{x.author} - {x.name} - {len(x)}" for x in collection_chunked])

In [ ]:
vectors = []

for b in collection_chunked:
    v = la.chunked_features(
        b.raw_source,
        n=len(b.raw_source)
    )

    v.insert(0, "Poem", b.name)
    v.insert(0, "Work", b.name.rsplit(" - ", 1)[0])
    v.insert(0, "Author", b.author)

    vectors.append(v)

collection_vect = pd.concat(vectors, ignore_index=True)

collection_vect.head()

In [ ]:
collection_vect.to_csv("output.csv", index=False, encoding="utf-8")